In [ ]:
%matplotlib inline
%precision 16
import numpy
import matplotlib.pyplot as plt

Before you turn this problem in, make sure everything runs as expected. First, restart the kernel (in the menubar, select Kernel $\rightarrow$ Restart) and then run all cells (in the menubar, select Cell $\rightarrow$ Run All).

Make sure you fill in any place that says YOUR CODE HERE or "YOUR ANSWER HERE", as well as your name and collaborators below:

# HW 5: Parabolic Problems

# Question 1 - Parabolic + Periodic BCs

Consider the temperature distribution in a thin circular ring with perimeter $2L$. Assume that the lateral surfaces are perfectly insulated, such that the temperature is distributed evenly throughout the ring cross-section and that the ends at $x=-L$ and $x=L$ are in perfect thermal contact.

This mathematical problem can be formulated as a heat equation with factor $k \in \mathbb R$ and periodic boundary conditions.
$$
    \frac{\partial u}{\partial t} = k \ \frac{\partial^2 u}{\partial x^2} \qquad \mbox{on } [-L,L]\times \mathbb R_+\\
    u(x,0) = u_0(x) ~~~~~~ u(-L,t) = u(L,t) ~~~~~ \frac{\partial}{\partial x} u(-L,t) = \frac{\partial}{\partial x} u(L,t).
$$

**(a)** [10] Solve the problem analytically.

YOUR ANSWER HERE

**(b)** [15] Write down the discretization for the problem resulting from using the Crank-Nicholson method showing the full matrix representation of the discretization and how you came to that representation.  Note that the periodicity will cause there to be a dependence that "wraps around" the domain so that the left most point in the domain depends on the right most point and vice-versa.

YOUR ANSWER HERE

**(b)** [10] Write a function to solve this problem using the Crank-Nicholson method.

In [1]:
def solve_heat_periodic_CN(m, kappa, L, t_0, t_final, U_0):
    """Solve the heat equation on a periodic domain using Crank-Nicholson
    
    :Input:
     - *m* (int) Number of points use to discretize the domain.  Note that
       the total number of points is *m+1*.
     - *kappa* (float) Diffusion coefficient
     - *L* (float) Length of half of the domain
     - *t_0* (float) Starting time
     - *t_final* (float) Time to integrate to
     - *U_0* (numpy.ndarray) Initial condition at time t_0, should be m+1
     
    :Output:
     - (numpy.ndarray) Solution at time t_final.  Note that this vector should m+1
    """

    # YOUR CODE HERE
    raise NotImplementedError()
    
    return U

In [2]:
L = numpy.pi
kappa = 0.075
u_true = lambda x, t: 6.0 * numpy.sin(2.0 * numpy.pi / L * x) * numpy.exp(-4.0 * kappa * (numpy.pi / L)**2 * t) + 3.0

# Discretization and output times
# Note that there are m unknowns including one of the points either at 0 or L
m = 200
delta_x = (2.0 * L) / float(m + 1)
x = numpy.linspace(-L, L, m + 2)
x_fine = numpy.linspace(-L, L, 100)
output_times = (0.0, 2.5, 5.0, 7.5, 10.0)

# Solve
U = numpy.empty((len(output_times), m + 1))
U[0, :] = u_true(x[:-1], 0.0)
for (n, t) in enumerate(output_times[1:]):
    U[n + 1, :] = solve_heat_periodic_CN(m, kappa, L, output_times[n], t, U[n, :])

error = numpy.linalg.norm(delta_x * (U[-1, :] - u_true(x[:-1], output_times[-1])), ord=1)
print("Error = %s" % error)
assert error < 1e-3
print("Success!")

# Plot some of the results
colors = ['k', 'r', 'b', 'g', 'c']
fig = plt.figure()
axes = fig.add_subplot(1, 1, 1)
for (n, t) in enumerate(output_times):
    axes.plot(x_fine, u_true(x_fine, t), 'k-')
    axes.plot(x[:-1], U[n, :], "o%s" % colors[n], label='t=%s' % numpy.round(t, 4))
    axes.set_xlabel("x")
    axes.set_ylabel("$U_(x,t)$")
    axes.set_title("Solution to Heat Equation")
    axes.set_xlim([-L, L])
    axes.legend()
plt.show()

NameError: name 'numpy' is not defined

# Question 2 - Diffusion and Decay

Consider the partial differential equation 

$$
    \frac{\partial u}{\partial t} = \kappa \frac{\partial^2 u}{\partial x^2} -\gamma u,
$$

which models a diffusion with parameter $\kappa \in \mathbb R_{>0}$ with decay specified by $\gamma \in \mathbb R_{>0}$.

Consider a $\theta$-method of the form

$$
    U_j^{n+1} = U_j^n + \frac {\Delta t} {2\Delta x^2} \left( U_{j-1}^n - 2 U_j^n + U_{j+1}^n + U_{j-1}^{n+1} - 2U_j^{n+1}+U_{j+1}^{n+1} \right)-\Delta t\gamma\left( (1-\theta) U^n_j +\theta U_j^{n+1} \right)
$$

where $\theta$ is a parameter.

**(a)** [10] Assume $\kappa = 1 $, by computing the local truncation error, show that this method is $p$th order accurate in time and second order accurate in space, where $p=2$ for $\theta = 0.5$ and $p=1$ otherwise.

YOUR ANSWER HERE

**(b)** [10] Using von Neumann analysis, show that this method is unconditionally stable if $\theta \geq 0.5$.

YOUR ANSWER HERE

**(c)** [10] Show that, if $\theta = 0$, then the method  is stable provided $\Delta t \leq \frac 2 \gamma$, independent of $\Delta x$.

YOUR ANSWER HERE

## Question 3 - Heat Equation

Consider the heat equation with zero boundary conditions:

$$
    \frac{\partial u}{\partial t} = \kappa \frac{\partial^2 u}{\partial x^2} \qquad \mbox{on } [0,L]\times \mathbb R_+\\
    u(x,0) = u_0(x) ~~~~~~ u(0,t) = 0 ~~~~~ u(L,t) = 0.
$$

**(a)** [20] Implement the *backward Euler* and *Crank-Nicholson* methods to solve this equation.  You are responsible for picking a $\Delta t$ so that the numerical method has enough accuracy but hits the end time $t_f$ exactly (or as much as this is possible with floating point arithmetic).  Note that making $\Delta t$ too small may harm your accuracy as much as picking one that is too big.

In [3]:
def solve_heat_BE(m, kappa, L, t_0, t_final, U_0, g_0, g_1):
    """Solve the heat equation using backward Euler and given boundaries
    
    Function will take the necessary number of time steps to reach t_final
    from t_0.  Note that this may mean taking a slightly smaller time step 
    size to hit t_f exactly.
    
    :Input:
     - *m* (int) Number of points use to discretize the domain.  Note that
       the total number of points is *m+2*, which includes the boundaries.
     - *kappa* (float) Diffusion coefficient
     - *L* (float) Length of the domain
     - *t_0* (float) Starting time
     - *t_final* (float) Time to integrate to
     - *U_0* (numpy.ndarray) Initial condition at time t_0, should be m+2
     - *g_0* (func) Boundary condition at x=0
     - *g_1* (func) Boundary condition at x=L
     
    :Output:
     - (numpy.ndarray) Solution at time t_final.  Note that this vector should m+2
       and include the boundary conditions.
    """
    
    delta_x = L / float(m + 1)
    x = numpy.linspace(0.0, L, m + 2)

    # YOUR CODE HERE
    raise NotImplementedError()
    
    return U

In [4]:
beta = 150.0
kappa = 0.02
L = 1.0
u_true = lambda x, t: numpy.exp(-(x - 0.4)**2. / (4.0 * kappa * t + 1.0 / beta)) \
                                            / numpy.sqrt(4.0 * kappa * beta * t + 1.0)

# Note that these will not be precisely zero
g_0 = lambda t: u_true(0.0, t)
g_1 = lambda t: u_true(L, t)

# Discretization and output times
m = 80
x = numpy.linspace(0.0, L, m + 2)
x_fine = numpy.linspace(0.0, L, 100)
delta_x = L / float(m + 1)
output_times = (0.0, 0.5, 1.0, 1.5, 2.0)

# Solve
U_BE = numpy.empty((len(output_times), m + 2))
U_BE[0, :] = u_true(x, 0.0)
for (n, t) in enumerate(output_times[1:]):
    U_BE[n + 1, :] = solve_heat_BE(m, kappa, L, output_times[n], t, U_BE[n, :], g_0, g_1)

error = numpy.linalg.norm(delta_x * (U_BE[-1, :] - u_true(x, output_times[-1])), ord=1)
print("Error BE = %s" % error)
assert error < 1e-3

# Plot some of the results
colors = ['k', 'r', 'b', 'g', 'c']
fig = plt.figure()
axes_BE = fig.add_subplot(1, 1, 1)
for (n, t) in enumerate(output_times):
    axes_BE.plot(x_fine, u_true(x_fine, t), 'k-')
    axes_BE.plot(x, U_BE[n, :], "o%s" % colors[n], label='t=%s' % numpy.round(t, 4))
    axes_BE.set_xlabel("x")
    axes_BE.set_ylabel("$U_(x,t)$")
    axes_BE.set_title("Solution to Heat Equation")
    axes_BE.set_xlim([0, L])
    axes_BE.legend()
    
plt.tight_layout()
plt.show()

NameError: name 'numpy' is not defined

In [ ]:
def solve_heat_CN(m, kappa, L, t_0, t_final, U_0, g_0, g_1):
    """Solve the heat equation using Crank-Nicholson and given boundaries
    
    Function will take the necessary number of time steps to reach t_final
    from t_0.  Note that this may mean taking a slightly smaller time step 
    size to hit t_f exactly.
    
    :Input:
     - *m* (int) Number of points use to discretize the domain.  Note that
       the total number of points is *m+2*, which includes the boundaries.
     - *kappa* (float) Diffusion coefficient
     - *L* (float) Length of the domain
     - *t_0* (float) Starting time
     - *t_final* (float) Time to integrate to
     - *U_0* (numpy.ndarray) Initial condition at time t_0, should be m+2
     - *g_0* (func) Boundary condition at x=0
     - *g_1* (func) Boundary condition at x=L
     
    :Output:
     - (numpy.ndarray) Solution at time t_final.  Note that this vector should m+2
       and include the boundary conditions.
    """
    
    delta_x = L / float(m + 1)
    x = numpy.linspace(0.0, L, m + 2)

    # YOUR CODE HERE
    raise NotImplementedError()
    
    return U

In [ ]:
beta = 150.0
kappa = 0.02
L = 1.0
u_true = lambda x, t: numpy.exp(-(x - 0.4)**2. / (4.0 * kappa * t + 1.0 / beta)) \
                                            / numpy.sqrt(4.0 * kappa * beta * t + 1.0)

# Note that these will not be precisely zero
g_0 = lambda t: u_true(0.0, t)
g_1 = lambda t: u_true(L, t)

# Discretization and output times
m = 20
x = numpy.linspace(0.0, L, m + 2)
x_fine = numpy.linspace(0.0, L, 100)
delta_x = L / float(m + 1)
output_times = (0.0, 0.5, 1.0, 1.5, 2.0)

# Solve
U_CN = numpy.empty((len(output_times), m + 2))
U_CN[0, :] = u_true(x, 0.0)
for (n, t) in enumerate(output_times[1:]):
    U_CN[n + 1, :] = solve_heat_CN(m, kappa, L, output_times[n], t, U_CN[n, :], g_0, g_1)

error = numpy.linalg.norm(delta_x * (U_CN[-1, :] - u_true(x, output_times[-1])), ord=1)
print(("Error CN = %s" % error))
assert error < 1e-3
print("Success!")

# Plot some of the results
colors = ['k', 'r', 'b', 'g', 'c']
fig = plt.figure()
axes_CN = fig.add_subplot(1, 1, 1)
for (n, t) in enumerate(output_times):
    axes_CN.plot(x_fine, u_true(x_fine, t), 'k-')
    axes_CN.plot(x, U_CN[n, :], "o%s" % colors[n], label='t=%s' % numpy.round(t, 4))
    axes_CN.set_xlabel("x")
    axes_CN.set_ylabel("$U_(x,t)$")
    axes_CN.set_title("Solution to Heat Equation")
    axes_CN.set_xlim([0, L])
    axes_CN.legend()

plt.tight_layout()
plt.show()

**(b)** [5] Perform a convergence analysis on the methods above (i.e. plot their convergence).

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

**(c)** [10] Are there initial and/or boundary conditions that you would expect would not be well handled in the methods above?  Justify your solution with the appropriate analysis and numerical experiment.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()